# Chapter 9 companion: the modified z-transform and closed-loop sampled systems.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.

Convention (as in the chapter): F(z,m) = sum_k f((k+m)T) z^-(k+1), 0 <= m < 1,
the transform of f delayed by (1-m)T.

Reproduces:
  * Example A1: f = e^-t, F(z,m) = e^{-mT}/(z - e^{-T}), and the two ends m = 0, m -> 1;
  * Example A2: ZOH + 1/(s+1), unit step: c((k+m)T) = 1 - e^{-(k+m)T};
  * Example A3: a delay of 0.4T (m = 0.6), T = 1 s:
    G(z,m) = (0.4512z + 0.1809)/(z(z - 0.3679)), step samples 0, 0.4512, 0.7981, 0.9257;
  * two samplers that are not synchronized: G1(z,0.2), G2(z,0.8), c(T) = 0.000398;
  * Example B1: ZOH + 1/(s+1) in a unity loop, T = ln 2: C/R = 0.5/z, checked by
    simulating the sampled loop with the plant integrated exactly;
  * Example B3 (two samplers): C/R = G1G2/(1 + G1G2 + HG2(z)), checked the same way
    with G1 = ZOH + 1/(s+1), G2 = ZOH + 1/(s+2), H = 1/(s+3).

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
# In the browser (JupyterLite), hide a harmless Matplotlib warning from its in-browser backend.
import sys, warnings
if "pyodide" in sys.modules:
    warnings.filterwarnings("ignore", message=r"The \w+ parameter as float was deprecated")

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import signal
from scipy.linalg import expm

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)

In [ ]:
def zoh(num, den, T):
    """ZOH equivalent (numerator, denominator in powers of z) from one matrix exponential."""
    A, B, C, D = signal.tf2ss(num, den)
    n = A.shape[0]
    M = np.zeros((n + 1, n + 1))
    M[:n, :n], M[:n, n:] = A, B
    E = expm(M * T)
    numz, denz = signal.ss2tf(E[:n, :n], E[:n, n:], C, D)
    return numz[0], denz

In [ ]:
def series(num, den, n):
    """First n coefficients of num(z)/den(z) in powers of z^-1 (same length vectors)."""
    num = np.r_[np.zeros(len(den) - len(num)), num]
    return signal.lfilter(num, den, np.r_[1.0, np.zeros(n - 1)])

In [ ]:
# ---------------------------------------------------------------- Example A1
T = 0.7

In [ ]:
z = 1.5 * np.exp(0.6j)
k = np.arange(400)
print("Example A1: f(t) = e^-t, T = 0.7 s, at z =", np.round(z, 4))

In [ ]:
for m in (0.0, 0.3, 0.8):
    s = np.sum(np.exp(-(k + m) * T) * z ** (-(k + 1.0)))
    print(f"  m = {m}: series {s:.6f},  e^(-mT)/(z - e^-T) = {np.exp(-m * T) / (z - np.exp(-T)):.6f}")

In [ ]:
F = z / (z - np.exp(-T))
print(f"  m = 0 gives F(z)/z = {F / z:.6f};  m -> 1 gives F(z) - f(0) = {F - 1:.6f}"
      f" = e^-T/(z - e^-T) = {np.exp(-T) / (z - np.exp(-T)):.6f}")

In [ ]:
# ---------------------------------------------------------------- Example A2
T = 1.0

In [ ]:
p = np.exp(-T)
print("\nExample A2: ZOH + 1/(s+1), unit step, T = 1 s: C(z,m) = 1/(z-1) - e^(-mT)/(z - e^-T)")

In [ ]:
for m in (0.25, 0.5):
    num = np.r_[0, (1 - np.exp(-m * T)), np.exp(-m * T) - p]
    c = series(num, np.polymul([1, -1], [1, -p]), 6)[1:]                # coefficients of z^-(k+1)
    print(f"  m = {m}: c((k+m)T) = {np.round(c, 4)};  1 - e^-(k+m)T = "
          f"{np.round(1 - np.exp(-(np.arange(5) + m) * T), 4)}")

In [ ]:
# ---------------------------------------------------------------- Example A3
m = 0.6

In [ ]:
num = [1 - np.exp(-m * T), np.exp(-m * T) - p]
den = np.polymul([1, 0], [1, -p])
print(f"\nExample A3: delay 0.4T, m = {m}: G(z,m) = ({num[0]:.4f} z + {num[1]:.4f})/(z (z - {p:.4f}))")
print(f"  DC gain G(1,m) = {np.polyval(num, 1) / np.polyval(den, 1):.4f};  zero at z = {-num[1] / num[0]:.3f}")
cs = signal.lfilter(np.r_[0, num], den, np.ones(6))
print("  step samples:", np.round(cs[:4], 4), "  1 - e^-(k - 0.4):",
      np.round(np.r_[0, 1 - np.exp(-(np.arange(1, 4) - 0.4))], 4))

In [ ]:
# ---------------------------------------------------------------- nonsynchronous samplers
T, h = 0.05, 0.2

In [ ]:
p = np.exp(-T)
g1 = [1 - np.exp(-h * T), np.exp(-h * T) - p]                # G1(z, h) numerator over z(z - p)
mm = 1 - h
g2 = [mm * T, (1 - mm) * T]                                  # (z-1)/z * T(mz + 1 - m)/(z-1)^2 -> over z(z-1)
print(f"\nTwo samplers, T = {T} s, offset hT = {h * T:.2f} s")
print(f"  G1(z,0.2) = ({g1[0]:.5f} z + {g1[1]:.5f})/(z (z - {p:.5f}))")
print(f"  G2(z,0.8) = ({g2[0]:.2f} z + {g2[1]:.2f})/(z (z - 1))")
print(f"  c(T) = {g2[0] * g1[0]:.6f}  (a(0.01) = {1 - np.exp(-0.01):.5f} held for 0.04 s:"
      f" {0.04 * (1 - np.exp(-0.01)):.6f})")

In [ ]:
# ---------------------------------------------------------------- closed loops, simulated
def step_matrices(A, B, T):
    n = A.shape[0]
    M = np.zeros((n + B.shape[1], n + B.shape[1]))
    M[:n, :n], M[:n, n:] = A, B
    E = expm(M * T)
    return E[:n, :n], E[:n, n:]

In [ ]:
# Example B1: e(k) = r - c(k), plant 1/(s+1) driven by the held error
T = np.log(2)

In [ ]:
Phi, Gam = step_matrices(np.array([[-1.0]]), np.array([[1.0]]), T)
x = np.zeros(1)
cB1 = []

In [ ]:
for kk in range(8):
    cB1.append(x[0])
    x = Phi @ x + Gam @ np.array([1.0 - x[0]])

In [ ]:
print("\nExample B1, T = ln 2: simulated c(kT) =", np.round(cB1, 4), "  (C/R = 0.5/z: 0, 0.5, 0.5, ...)")

In [ ]:
# Example B3: E1 = R - C, E2 = G1 E1* - H C, C = G2 E2*  (H acts on the continuous C)
T = 0.3

In [ ]:
A = np.array([[-1.0, 0, 0], [0, -2.0, 0], [0, 1.0, -3.0]])   # x1: G1 plant, xc: G2 plant, xh: H
B = np.array([[1.0, 0], [0, 1.0], [0, 0]])
Phi, Gam = step_matrices(A, B, T)
x = np.zeros(3)
c_sim = []

In [ ]:
for kk in range(25):
    c_sim.append(x[1])
    e1 = 1.0 - x[1]
    e2 = x[0] - x[2]
    x = Phi @ x + Gam @ np.array([e1, e2])

In [ ]:
n1, d1 = zoh([1], [1, 1], T)
n2, d2 = zoh([1], [1, 2], T)
nh, dh = zoh([1], np.polymul([1, 2], [1, 3]), T)                 # HG2(z) = Z[ZOH H(s) /(s+2)]

In [ ]:
# C/R = G1G2 / (1 + G1G2 + HG2) with every term over a common denominator
N12, D12 = np.polymul(n1, n2), np.polymul(d1, d2)

In [ ]:
num = np.polymul(N12, dh)
den = np.polyadd(np.polyadd(np.polymul(D12, dh), np.polymul(N12, dh)), np.polymul(nh, D12))
num = np.r_[np.zeros(len(den) - len(num)), num]                  # align powers of z
c_z = signal.lfilter(num, den, np.ones(25))
print("Example B3 (T = 0.3 s), unit step:")
print("  simulated loop      :", np.round(c_sim[:8], 5))
print("  G1G2/(1+G1G2+HG2(z)):", np.round(c_z[:8], 5))
print("  largest difference over 25 samples:", f"{np.max(np.abs(np.array(c_sim) - c_z)):.1e}")

In [ ]:
# ---------------------------------------------------------------- figure
# left: Example A2, sweeping m traces the output between the samples
T, p = 1.0, np.exp(-1.0)

In [ ]:
tt, cc = [], []

In [ ]:
for m in np.linspace(0, 0.99, 34):
    num = np.r_[0, 1 - np.exp(-m * T), np.exp(-m * T) - p]
    c = series(num, np.polymul([1, -1], [1, -p]), 8)        # c[k+1] = c((k+m)T)
    for kk in range(6):
        tt.append((kk + m) * T)
        cc.append(c[kk + 1])

In [ ]:
tc = np.linspace(0, 6, 600)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(tc, 1 - np.exp(-tc), "k", lw=1, label="1 - e^-t")
axes[0].plot(tt, cc, ".", ms=3, color="C0", label="C(z,m), m swept")
axes[0].plot(np.arange(7), 1 - np.exp(-np.arange(7.0)), "o", color="C3", label="samples")
axes[0].set(xlabel="t (s)", title="Example A2: between the samples")
axes[0].legend(fontsize=8)

In [ ]:
# right: Example B3, simulated loop against the sampled-graph formula
axes[1].plot(0.3 * np.arange(25), c_sim, "o", ms=4, label="simulated loop")

In [ ]:
axes[1].plot(0.3 * np.arange(25), c_z, "x", ms=5, label="G1G2/(1 + G1G2 + HG2)")
axes[1].set(xlabel="t (s)", title="Example B3 with two samplers, T = 0.3 s")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "ch09_modified_z.png", dpi=150)
print("\nSaved figures/ch09_modified_z.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**A delay that is not a whole period.** A zero-order hold drives $1/(s+1)$, and the output
reaches the sampler $\Delta T$ seconds late, $0 < \Delta < 1$. The modified $z$-transform gives
$G(z,m)$ with $m = 1-\Delta$. Move $\Delta$ and watch the samples $c(kT)$ slide along the
undelayed curve $1 - e^{-t}$: they read it at $(k-\Delta)T$. On the right, the delay adds a pole
at $z = 0$ and a zero that moves with $m$; the DC gain stays 1.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"


def explore(T=1.0, Delta=0.4):
    m = 1.0 - Delta
    p = np.exp(-T)
    b1 = 1.0 - np.exp(-m * T)                 # G(z,m) = (b1 z + b0) / (z (z - e^{-T}))
    b0 = np.exp(-m * T) - p

    # unit step through G(z,m): c(k) = p c(k-1) + b1 u(k-1) + b0 u(k-2), u = 1 for k >= 0
    t_end = max(5.0, 8 * T)
    n = int(np.floor(t_end / T + 1e-9)) + 1
    c = np.zeros(n)
    for k in range(1, n):
        c[k] = p * c[k - 1] + b1 + (b0 if k >= 2 else 0.0)
    k = np.arange(n)
    closed = np.where(k >= 1, 1 - np.exp(-(k - Delta) * T), 0.0)    # the curve read Delta*T late

    t = np.linspace(0, t_end, 800)
    c_delayed = np.where(t >= Delta * T, 1 - np.exp(-(t - Delta * T)), 0.0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.0), gridspec_kw={"width_ratios": [2.1, 1]})
    ax1.plot(t, 1 - np.exp(-t), "--", color=GRAY, lw=1.4, label="no delay: $1 - e^{-t}$")
    ax1.plot(t, c_delayed, color=TEAL, lw=1.6, label=f"delayed output, $\\Delta T$ = {Delta * T:.2f} s")
    ax1.vlines(k * T, 0, c, color=ORANGE, lw=1)
    ax1.plot(k * T, c, "o", color=ORANGE, ms=6, label="samples $c(kT)$ from $G(z,m)$")
    ax1.set_xlim(0, t_end)
    ax1.set_ylim(-0.05, 1.15)
    ax1.set_xlabel("t (s)")
    ax1.set_ylabel("c")
    ax1.set_title(f"T = {T:g} s, delay $\\Delta$ = {Delta:.2f} T, so m = {m:.2f}", fontsize=11)
    ax1.legend(fontsize=10, loc="lower right")

    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.plot([0, p], [0, 0], "x", color=BLUE, ms=11, mew=2.5, label="poles")
    zero = -b0 / b1
    if abs(zero) <= 1.25:
        ax2.plot([zero], [0], "o", mfc="white", mec=ORANGE, mew=2.5, ms=10, label="zero")
    else:
        ax2.annotate(f"zero at {zero:.2f}", xy=(-1.28, 0), xytext=(-1.2, 0.45), color=ORANGE,
                     fontsize=10, arrowprops=dict(arrowstyle="->", color=ORANGE))
    ax2.axhline(0, color="k", lw=0.5)
    ax2.axvline(0, color="k", lw=0.5)
    ax2.set_xlim(-1.3, 1.3)
    ax2.set_ylim(-1.3, 1.3)
    ax2.set_aspect("equal")
    ax2.set_xlabel("Re z")
    ax2.set_ylabel("Im z")
    ax2.set_title("poles and zero of G(z,m)", fontsize=11)
    ax2.legend(fontsize=10, loc="lower left")
    fig.tight_layout()

    dc = (b1 + b0) / (1 - p)
    print(f"G(z,m) = ({b1:.4f} z + {b0:.4f}) / (z (z - {p:.4f})),  m = 1 - Delta = {m:.2f}")
    print(f"zero at z = {zero:.3f};  DC gain G(1,m) = {dc:.4f}")
    print("c(kT), k = 0..4:", np.round(c[:5], 4),
          f"  (equal to 1 - e^-(k - Delta)T: largest difference {np.max(np.abs(c - closed)):.1e})")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             T=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description="T (s)",
                           continuous_update=False),
             Delta=FloatSlider(value=0.4, min=0.05, max=0.95, step=0.05, description="Δ (of T)",
                               continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 9.** These cells go with the Hands-on exercises at the end of
Chapter 9. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
def integrator_loop(T, t0, K, n=40, m=200, r=1.0):
    """Conveyor y' = u; u_k = K (r - y(kT)) reaches the hold at kT + t0 (0 <= t0 <= T).
    y(t) is piecewise linear, so it is computed exactly at its corners and interpolated."""
    corners, values = [0.0], [0.0]
    y, u_old = 0.0, 0.0                            # y(kT); command on the hold before kT + t0
    for k in range(n):
        u_new = K * (r - y)
        y_mid = y + u_old * t0                     # at kT + t0, the new command takes over
        y = y_mid + u_new * (T - t0)               # at (k+1)T
        corners += [k * T + t0, (k + 1) * T]
        values += [y_mid, y]
        u_old = u_new
    t = np.linspace(0, n * T, n * m + 1)
    return t, np.interp(t, corners, values)

In [ ]:
# ---------------------------------------------------------------- H1
def h1_gateway(h=0.25, T=0.04):
    """H1: the gateway re-reads the held command at (k+h)T and holds it again."""
    n = 8
    t = np.linspace(0, n * T, 2000)
    u_ctrl = np.ones_like(t)                         # unit step command from the controller
    u_gate = np.where(t >= h * T, 1.0, 0.0)          # what the motor receives
    y = np.maximum(t - h * T, 0.0)                   # position, m (u = 1 m/s)
    k = np.arange(1, n)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.6))
    ax1.step(t, u_ctrl, where="post", color=GRAY, lw=1.4, label="controller D/A")
    ax1.step(t, u_gate, where="post", color=ORANGE, lw=1.4, label="gateway D/A")
    ax1.set_ylim(-0.1, 1.3)
    ax1.set_xlabel("t (s)")
    ax1.legend(loc="lower right", fontsize=9)
    ax1.set_title(f"h = {h:.2f}: delay {1000 * h * T:.0f} ms", fontsize=11)
    ax2.plot(t, y, color=BLUE, lw=1.4, label="position y(t)")
    ax2.plot(k * T, np.maximum(k * T - h * T, 0), "o", color=ORANGE, ms=6, label="samples y(kT)")
    ax2.plot(k * T, k * T, "x", color=TEAL, ms=7, label="h = 0")
    ax2.set_xlabel("t (s)")
    ax2.legend(loc="upper left", fontsize=9)
    ax2.set_title(f"y(T) = {k[0] * T - h * T:.3f} m", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_which_m(t0_frac=0.25):
    """H2: pump 1/(s+1), T = ln 2, computation delay t0 = t0_frac T; samples of the step response."""
    T = np.log(2)
    t0 = t0_frac * T
    tt = np.linspace(0, 6 * T, 1200)
    c = np.where(tt > t0, 1 - np.exp(-(tt - t0)), 0.0)
    k = np.arange(7)
    ck = np.where(k * T > t0, 1 - np.exp(-(k * T - t0)), 0.0)
    m_right, m_wrong = 1 - t0_frac, t0_frac
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(tt, c, color=GRAY, lw=1.4, label="c(t)")
    ax.plot(k * T, ck, "o", color=ORANGE, ms=7, label=f"samples: model with m = {m_right:.2f}")
    wrong = np.where(k * T > (1 - t0_frac) * T, 1 - np.exp(-(k * T - (1 - t0_frac) * T)), 0.0)
    ax.plot(k * T, wrong, "s--", color=TEAL, ms=5, lw=1, label=f"model with m = {m_wrong:.2f}")
    ax.axvline(t0, color=BLUE, ls=":", lw=1)
    ax.set_xlabel("t (s)")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_title(f"t0 = {t0_frac:.2f} T: c(1) = {ck[1]:.3f}", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_deadbeat_delay(t0_ms=10.0):
    """H3: deadbeat conveyor loop (K = 1/T, T = 40 ms) with computation time t0."""
    T = 0.04
    t0 = t0_ms / 1000
    t, y = integrator_loop(T, t0, 1 / T, n=15)
    D = t0 / T
    poles = np.roots([1, -D, D])
    k = np.arange(16)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(t, y, color=GRAY, lw=1.4, label="y(t)")
    ax1.plot(k * T, y[k * 200], "o", color=ORANGE, ms=5, label="samples")
    ax1.axhline(1, color="k", lw=0.6)
    ax1.set_ylim(-0.1, 2.0)
    ax1.set_xlabel("t (s)")
    ax1.legend(loc="upper right", fontsize=9)
    ax1.set_title(f"t0 = {t0_ms:g} ms: sample peak {y[k * 200].max():.3f}, true peak {y.max():.3f}",
                  fontsize=11)
    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.plot(poles.real, poles.imag, "x", color=ORANGE, ms=12, mew=2.5)
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.3, 1.3)
    ax2.set_ylim(-1.3, 1.3)
    ax2.set_title(f"|z| = {max(abs(poles)):.3f}", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_choose_T(T_ms=40.0, t0_ms=5.0):
    """H4: fixed computation time t0, K = 1/T; true peak and the error envelope at 0.2 s."""
    T, t0 = T_ms / 1000, t0_ms / 1000
    n = int(np.ceil(0.4 / T)) + 2
    t, y = integrator_loop(T, t0, 1 / T, n=n)
    D = t0 / T
    rz = max(abs(np.roots([1, -D, D])))
    env = rz ** (0.2 / T)
    ok = (y.max() - 1 <= 0.10 + 1e-9) and env <= 0.02
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t, y, color=ORANGE, lw=1.4, label="y(t)")
    ax.axhline(1.1, color=TEAL, ls="--", lw=1, label="10% overshoot")
    ax.axvline(0.2, color=GRAY, ls=":", lw=1)
    ax.set_xlim(0, 0.4)
    ax.set_ylim(-0.1, 1.5)
    ax.set_xlabel("t (s)")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_title(f"T = {T_ms:g} ms: true overshoot {100 * (y.max() - 1):.1f}%, envelope at 0.2 s "
                 f"{env:.4f}: {'PASS' if ok else 'FAIL'}", fontsize=11)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    interact(h1_gateway, h=FloatSlider(value=0.25, min=0.0, max=0.95, step=0.05, description="h",
                                       continuous_update=False),
             T=FloatSlider(value=0.04, min=0.01, max=0.1, step=0.01, description="T (s)",
                           continuous_update=False))
    interact(h2_which_m, t0_frac=FloatSlider(value=0.25, min=0.0, max=0.95, step=0.05,
                                             description="t0 / T", continuous_update=False))
    interact(h3_deadbeat_delay, t0_ms=FloatSlider(value=10, min=0, max=40, step=1,
                                                  description="t0 (ms)", continuous_update=False))
    interact(h4_choose_T, T_ms=FloatSlider(value=40, min=10, max=100, step=1, description="T (ms)",
                                           continuous_update=False),
             t0_ms=FloatSlider(value=5, min=0, max=10, step=0.5, description="t0 (ms)",
                               continuous_update=False))
except ImportError:
    h1_gateway()
    h2_which_m()
    h3_deadbeat_delay()
    h4_choose_T()